# Preparacion de datos para un sistema inteligente de priorizacion de cobranzas

Este notebook convierte `01_generacion/data/features_ml.csv` en artefactos listos para modelado. Sigue las decisiones del EDA: split por `factura_id`, tratamiento explicito de nulos estructurales, conservacion de outliers y exclusion de identificadores/target de los predictores.


## 1. Librerias y configuracion

Se definen rutas relativas al proyecto, columnas esperadas y grupos de variables usados por el pipeline.


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler
from sklearn.utils.class_weight import compute_class_weight

TARGET_COL = "target_mora"
RAW_TARGET_COL = "target"
TARGET_CLASS_ORDER = ["on_time", "+30", "+60", "+90"]
VALID_TARGET_CLASSES = set(TARGET_CLASS_ORDER)
VALID_CONDICION_DIAS = {30, 45, 60, 90}
RATE_COLUMNS = ["tasa_cumplimiento", "tasa_contacto_cliente", "tasa_cumpl_promesas"]
ID_AND_LABEL_COLUMNS = ["factura_id", "cliente_id", "fecha_corte", TARGET_COL]
SIMULATION_ONLY_COLUMNS = ["perfil_pago"]

SECTOR_COLUMNS = [
    "sector_retail", "sector_manufactura", "sector_servicios", "sector_construccion",
    "sector_agro", "sector_tecnologia", "sector_salud", "sector_transporte",
]

REQUIRED_COLUMNS = [
    "factura_id", "cliente_id", "num_corte", "fecha_corte", "monto", "condicion_dias",
    "antiguedad_meses", "tiene_garantia", "num_facturas_prev", "mora_promedio_hist",
    "mora_ultimo_tramo", "tasa_cumplimiento", "monto_promedio_hist", "ratio_monto",
    "moras_consecutivas", "num_gestiones_factura", "dias_desde_ultima_gestion",
    "dias_desde_emision", "dias_hasta_vence", "tasa_contacto_cliente", "ultimo_resultado_enc",
    "num_no_contesta_cons", "tiene_disputa_activa", "tiene_promesa_activa",
    "num_promesas_rotas", "tasa_cumpl_promesas", "promesas_total", TARGET_COL,
    *SECTOR_COLUMNS,
]

LOG_SKEWED_COLS = [
    "monto", "monto_promedio_hist", "ratio_monto", "mora_promedio_hist",
    "mora_ultimo_tramo", "num_gestiones_factura", "dias_hasta_vence_pos",
    "dias_mora_observable", "num_no_contesta_cons", "num_promesas_rotas",
    "promesas_total", "dias_transcurridos_corte",
]

BINARY_COLS = [
    "tiene_garantia", "tiene_disputa_activa", "tiene_promesa_activa",
    "sin_gestion_previa", "esta_vencida_al_corte", "cliente_nuevo", *SECTOR_COLUMNS,
]

NUMERIC_PLAIN_COLS = [
    "condicion_dias", "antiguedad_meses", "num_facturas_prev", "tasa_cumplimiento",
    "moras_consecutivas", "tasa_contacto_cliente", "tasa_cumpl_promesas",
    "friccion_contacto", "ratio_promesas_rotas", "intensidad_gestion",
    "dias_desde_ultima_gestion",
]

CATEGORICAL_COLS = ["ultimo_resultado_enc"]

CLUSTERING_CONT_COLS = [
    "monto", "condicion_dias", "antiguedad_meses",
    "mora_promedio_hist", "mora_ultimo_tramo", "monto_promedio_hist", "ratio_monto",
    "dias_desde_ultima_gestion", "dias_hasta_vence", "dias_mora_observable",
    "dias_hasta_vence_pos", "intensidad_gestion", "friccion_contacto",
]
CLUSTERING_RATIO_COLS = [
    "tasa_cumplimiento", "tasa_contacto_cliente", "tasa_cumpl_promesas", "ratio_promesas_rotas",
]
CLUSTERING_COUNT_COLS = [
    "num_facturas_prev", "moras_consecutivas", "num_gestiones_factura",
    "num_no_contesta_cons", "num_promesas_rotas", "promesas_total",
]
CLUSTERING_BINARY_COLS = [
    "tiene_disputa_activa", "tiene_promesa_activa", "sin_gestion_previa",
    "esta_vencida_al_corte", "cliente_nuevo",
]



def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_input = (candidate / "01_generacion" / "data" / "features_ml.csv").exists()
        has_phase = (candidate / "03_preparacion").exists()
        if has_input and has_phase:
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto desde el directorio actual.")


PROJECT_ROOT = find_project_root()
PHASE_DIR = PROJECT_ROOT / "03_preparacion"
DATA_PATH = PROJECT_ROOT / "01_generacion" / "data" / "features_ml.csv"
OUTPUT_DIR = PHASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Entrada:", DATA_PATH.relative_to(PROJECT_ROOT))
print("Salida:", OUTPUT_DIR.relative_to(PROJECT_ROOT))


Entrada: 01_generacion\data\features_ml.csv
Salida: 03_preparacion\outputs


## 2. Carga y validacion estructural

La validacion se concentra en reglas que bloquean modelado: columnas obligatorias, target unico por factura, dominios validos y ausencia de `perfil_pago` como predictor.


In [2]:
def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontro el archivo: {path}")

    df = pd.read_csv(path)
    if RAW_TARGET_COL in df.columns and TARGET_COL not in df.columns:
        df = df.rename(columns={RAW_TARGET_COL: TARGET_COL})

    missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Columnas requeridas faltantes: {missing}")

    df["fecha_corte"] = pd.to_datetime(df["fecha_corte"], errors="coerce")
    return df


def _safe_date(value) -> str | None:
    return None if pd.isna(value) else str(value.date())


def validate_dataset(df: pd.DataFrame) -> dict:
    target_by_factura = df.groupby("factura_id")[TARGET_COL].nunique(dropna=False)
    validations = {
        "shape": list(df.shape),
        "facturas_unicas": int(df["factura_id"].nunique()),
        "clientes_unicos": int(df["cliente_id"].nunique()),
        "fecha_min": _safe_date(df["fecha_corte"].min()),
        "fecha_max": _safe_date(df["fecha_corte"].max()),
        "columnas_requeridas_faltantes": sorted(set(REQUIRED_COLUMNS) - set(df.columns)),
        "duplicados_exactos": int(df.duplicated().sum()),
        "duplicados_factura_corte": int(df.duplicated(subset=["factura_id", "num_corte"]).sum()),
        "target_inconsistente_por_factura": int((target_by_factura > 1).sum()),
        "fechas_nulas": int(df["fecha_corte"].isna().sum()),
        "monto_no_positivo": int((df["monto"] <= 0).sum()),
        "antiguedad_no_positiva": int((df["antiguedad_meses"] <= 0).sum()),
        "condicion_invalida": int((~df["condicion_dias"].isin(VALID_CONDICION_DIAS)).sum()),
        "target_nulos": int(df[TARGET_COL].isna().sum()),
        "clases_target_invalidas": sorted(set(df[TARGET_COL].dropna().unique()) - VALID_TARGET_CLASSES),
        "nulos_por_columna": df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_dict(),
        "perfil_pago_presente": "perfil_pago" in df.columns,
    }
    for col in RATE_COLUMNS:
        validations[f"{col}_fuera_de_rango"] = int(((df[col] < 0) | (df[col] > 1)).sum())
    validations["filas_sector_invalido"] = int((df[SECTOR_COLUMNS].sum(axis=1) != 1).sum())
    return validations


def build_validation_checklist(validations: dict) -> pd.DataFrame:
    checks = [
        ("Columnas requeridas disponibles", not validations["columnas_requeridas_faltantes"], str(validations["columnas_requeridas_faltantes"])),
        ("Sin duplicados exactos", validations["duplicados_exactos"] == 0, validations["duplicados_exactos"]),
        ("Sin duplicados factura-corte", validations["duplicados_factura_corte"] == 0, validations["duplicados_factura_corte"]),
        ("Target unico por factura", validations["target_inconsistente_por_factura"] == 0, validations["target_inconsistente_por_factura"]),
        ("Fechas de corte validas", validations["fechas_nulas"] == 0, validations["fechas_nulas"]),
        ("Montos positivos", validations["monto_no_positivo"] == 0, validations["monto_no_positivo"]),
        ("Antiguedad positiva", validations["antiguedad_no_positiva"] == 0, validations["antiguedad_no_positiva"]),
        ("Condicion de pago valida", validations["condicion_invalida"] == 0, validations["condicion_invalida"]),
        ("Target completo", validations["target_nulos"] == 0, validations["target_nulos"]),
        ("Clases target validas", not validations["clases_target_invalidas"], str(validations["clases_target_invalidas"])),
        ("Sector one-hot valido", validations["filas_sector_invalido"] == 0, validations["filas_sector_invalido"]),
        ("Sin perfil_pago como predictor", not validations["perfil_pago_presente"], validations["perfil_pago_presente"]),
    ]
    for col in RATE_COLUMNS:
        checks.append((f"{col} en rango [0, 1]", validations[f"{col}_fuera_de_rango"] == 0, validations[f"{col}_fuera_de_rango"]))

    return pd.DataFrame([
        {"validacion": name, "estado": "Cumple" if ok else "Revisar", "evidencia": evidence}
        for name, ok, evidence in checks
    ])


def assert_validation_checklist(checklist: pd.DataFrame) -> None:
    failed = checklist[checklist["estado"] != "Cumple"]
    if not failed.empty:
        raise ValueError(f"Validaciones criticas fallidas: {failed.to_dict(orient='records')}")


In [3]:
df_raw = load_dataset(DATA_PATH)
validations = validate_dataset(df_raw)
validation_checklist = build_validation_checklist(validations)
assert_validation_checklist(validation_checklist)

validation_checklist


,validacion,estado,evidencia
0,Columnas requeridas disponibles,Cumple,[]
1,Sin duplicados exactos,Cumple,0
2,Sin duplicados factura-corte,Cumple,0
3,Target unico por factura,Cumple,0
4,Fechas de corte validas,Cumple,0
5,Montos positivos,Cumple,0
6,Antiguedad positiva,Cumple,0
7,Condicion de pago valida,Cumple,0
8,Target completo,Cumple,0
9,Clases target validas,Cumple,[]


## 3. Limpieza y revision de redundancias

Los nulos de corte inicial se codifican como ausencia estructural. Las redundancias deterministicas se eliminan solo cuando la equivalencia se confirma en los datos.


In [4]:
def clean_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    df["sin_gestion_previa"] = (
        df["dias_desde_ultima_gestion"].isna() & df["ultimo_resultado_enc"].isna()
    ).astype(int)

    df["dias_desde_ultima_gestion"] = df["dias_desde_ultima_gestion"].fillna(-1)
    df["ultimo_resultado_enc"] = (
        df["ultimo_resultado_enc"]
        .astype("Int64")
        .astype(str)
        .replace("<NA>", "sin_gestion_previa")
    )
    df["ultimo_resultado_enc"] = df["ultimo_resultado_enc"].map(
        lambda value: f"cod_{value}" if value != "sin_gestion_previa" else value
    )

    redundant = {
        "num_corte_eq_num_gestiones_factura": bool((df["num_corte"] == df["num_gestiones_factura"]).all()),
        "dias_hasta_vence_eq_condicion_menos_dias_desde_emision": bool(
            (df["dias_hasta_vence"] == (df["condicion_dias"] - df["dias_desde_emision"])).all()
        ),
    }

    cols_to_drop = []
    if redundant["num_corte_eq_num_gestiones_factura"]:
        cols_to_drop.append("num_corte")
    if redundant["dias_hasta_vence_eq_condicion_menos_dias_desde_emision"]:
        cols_to_drop.append("dias_desde_emision")

    df = df.drop(columns=cols_to_drop)
    summary = {
        "columnas_eliminadas": cols_to_drop,
        "nulos_restantes": df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_dict(),
        **redundant,
    }
    return df, summary


def build_redundancy_review(cleaning_summary: dict) -> pd.DataFrame:
    rows = [
        {
            "variable": "num_corte",
            "decision": "eliminar" if "num_corte" in cleaning_summary["columnas_eliminadas"] else "conservar",
            "motivo": "Equivale a num_gestiones_factura en esta version del dataset.",
        },
        {
            "variable": "dias_desde_emision",
            "decision": "eliminar" if "dias_desde_emision" in cleaning_summary["columnas_eliminadas"] else "conservar",
            "motivo": "Equivale a condicion_dias - dias_hasta_vence en esta version del dataset.",
        },
        {
            "variable": "ratio_monto",
            "decision": "conservar_transformar",
            "motivo": "Aporta escala relativa contra historial; se trata como sesgada con log1p y RobustScaler.",
        },
        {
            "variable": "mora_ultimo_tramo",
            "decision": "conservar_transformar",
            "motivo": "Complementa mora_promedio_hist con senal reciente; se controla con transformacion robusta.",
        },
        {
            "variable": "promesas_total",
            "decision": "conservar_transformar",
            "motivo": "Resume historial de promesas; se conserva junto a ratio_promesas_rotas para lectura operativa.",
        },
    ]
    return pd.DataFrame(rows)


In [5]:
df_clean, cleaning_summary = clean_dataset(df_raw)
redundancy_review = build_redundancy_review(cleaning_summary)

cleaning_summary


{'columnas_eliminadas': ['num_corte', 'dias_desde_emision'],
 'nulos_restantes': {},
 'num_corte_eq_num_gestiones_factura': True,
 'dias_hasta_vence_eq_condicion_menos_dias_desde_emision': True}

## 4. Feature engineering y outliers

Las nuevas variables son observables al corte. Los outliers se describen con IQR, pero no se eliminan porque pueden representar casos criticos de cobranza.


In [6]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dias_transcurridos_corte"] = df["condicion_dias"] - df["dias_hasta_vence"]
    df["esta_vencida_al_corte"] = (df["dias_hasta_vence"] < 0).astype(int)
    df["dias_mora_observable"] = np.maximum(0, -df["dias_hasta_vence"])
    df["dias_hasta_vence_pos"] = np.maximum(0, df["dias_hasta_vence"])
    df["cliente_nuevo"] = (df["num_facturas_prev"] == 0).astype(int)
    df["intensidad_gestion"] = df["num_gestiones_factura"] / np.maximum(df["dias_transcurridos_corte"] + 1, 1)
    df["friccion_contacto"] = df["num_no_contesta_cons"] / np.maximum(df["num_gestiones_factura"], 1)
    df["ratio_promesas_rotas"] = df["num_promesas_rotas"] / np.maximum(df["promesas_total"], 1)
    return df


def outlier_summary_iqr(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col].astype(float)
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        pct_out = ((s < lower) | (s > upper)).mean() * 100
        rows.append({
            "variable": col,
            "q1": round(float(q1), 4),
            "q3": round(float(q3), 4),
            "iqr": round(float(iqr), 4),
            "limite_inferior": round(float(lower), 4),
            "limite_superior": round(float(upper), 4),
            "pct_outliers_iqr": round(float(pct_out), 2),
            "asimetria": round(float(s.skew()), 4),
            "decision": "Conservar; tratar con log1p/RobustScaler si el modelo es sensible a escala",
        })
    return pd.DataFrame(rows).sort_values("pct_outliers_iqr", ascending=False)


def build_feature_list(df: pd.DataFrame) -> list[str]:
    excluded = set(ID_AND_LABEL_COLUMNS + SIMULATION_ONLY_COLUMNS)
    return [col for col in df.columns if col not in excluded]


In [7]:
df_prepared = engineer_features(df_clean)
feature_cols = build_feature_list(df_prepared)
outlier_cols = [col for col in LOG_SKEWED_COLS if col in feature_cols]
outlier_table = outlier_summary_iqr(df_prepared, outlier_cols)

print("Shape dataset preparado:", df_prepared.shape)
outlier_table.head(10)


Shape dataset preparado: (19671, 43)


,variable,q1,q3,iqr,limite_inferior,limite_superior,pct_outliers_iqr,asimetria,decision
0,monto,6641.3400,36312.9200,29671.5800,-37866.0300,80820.2900,7.88,1.8134,Conservar; tratar con log1p/RobustScaler si el...
6,dias_hasta_vence_pos,0.0000,33.0000,33.0000,-49.5000,82.5000,7.19,1.2774,Conservar; tratar con log1p/RobustScaler si el...
9,num_promesas_rotas,0.0000,4.0000,4.0000,-6.0000,10.0000,4.83,1.7725,Conservar; tratar con log1p/RobustScaler si el...
8,num_no_contesta_cons,0.0000,1.0000,1.0000,-1.5000,2.5000,4.66,2.1549,Conservar; tratar con log1p/RobustScaler si el...
7,dias_mora_observable,0.0000,30.0000,30.0000,-45.0000,75.0000,4.59,1.5213,Conservar; tratar con log1p/RobustScaler si el...
10,promesas_total,1.0000,8.0000,7.0000,-9.5000,18.5000,2.75,1.2944,Conservar; tratar con log1p/RobustScaler si el...
2,ratio_monto,0.5587,1.4885,0.9297,-0.8359,2.8831,1.48,6.1936,Conservar; tratar con log1p/RobustScaler si el...
1,monto_promedio_hist,7066.3100,42436.9300,35370.6200,-45989.6200,95492.8600,0.99,1.2307,Conservar; tratar con log1p/RobustScaler si el...
4,mora_ultimo_tramo,0.0000,35.0000,35.0000,-52.5000,87.5000,0.41,0.9675,Conservar; tratar con log1p/RobustScaler si el...
3,mora_promedio_hist,3.7500,42.1400,38.3900,-53.8350,99.7250,0.19,0.5571,Conservar; tratar con log1p/RobustScaler si el...


## 5. Split por factura

El split se estratifica a nivel `factura_id`, no a nivel fila. Esto evita que cortes de una misma factura queden en train y test.


In [8]:
def split_by_factura(df: pd.DataFrame, test_size: float = 0.20, random_state: int = 42):
    factura_target = df.groupby("factura_id", as_index=False)[TARGET_COL].first()
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(factura_target[["factura_id"]], factura_target[TARGET_COL]))

    train_ids = factura_target.iloc[train_idx].copy()
    test_ids = factura_target.iloc[test_idx].copy()
    train_facturas = set(train_ids["factura_id"])
    test_facturas = set(test_ids["factura_id"])

    if not train_facturas.isdisjoint(test_facturas):
        raise ValueError("Hay facturas compartidas entre train y test.")

    df_train = df[df["factura_id"].isin(train_facturas)].copy()
    df_test = df[df["factura_id"].isin(test_facturas)].copy()
    return df_train, df_test, train_ids, test_ids


def build_split_integrity_check(df: pd.DataFrame, train_ids: pd.DataFrame, test_ids: pd.DataFrame) -> pd.DataFrame:
    train_facturas = set(train_ids["factura_id"])
    test_facturas = set(test_ids["factura_id"])
    dataset_facturas = set(df["factura_id"])
    checks = [
        ("Sin facturas compartidas", len(train_facturas & test_facturas) == 0, len(train_facturas & test_facturas)),
        ("Todas las facturas asignadas", dataset_facturas == (train_facturas | test_facturas), len(dataset_facturas - (train_facturas | test_facturas))),
        ("Sin facturas extra en split", (train_facturas | test_facturas).issubset(dataset_facturas), len((train_facturas | test_facturas) - dataset_facturas)),
    ]
    return pd.DataFrame([
        {"validacion": name, "estado": "Cumple" if ok else "Revisar", "evidencia": evidence}
        for name, ok, evidence in checks
    ])


def build_target_distribution(df_train: pd.DataFrame, df_test: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split_name, split_df in {"train": df_train, "test": df_test}.items():
        counts = split_df[TARGET_COL].value_counts().reindex(TARGET_CLASS_ORDER, fill_value=0)
        for target_class, count in counts.items():
            rows.append({
                "split": split_name,
                "target_mora": target_class,
                "filas": int(count),
                "porcentaje_filas": round(float(count / len(split_df) * 100), 4),
                "facturas": int(split_df.loc[split_df[TARGET_COL] == target_class, "factura_id"].nunique()),
            })
    return pd.DataFrame(rows)


In [9]:
df_train, df_test, train_ids, test_ids = split_by_factura(df_prepared)
split_integrity_check = build_split_integrity_check(df_prepared, train_ids, test_ids)
assert_validation_checklist(split_integrity_check)

X_train = df_train[feature_cols].copy()
X_test = df_test[feature_cols].copy()
y_train = df_train[TARGET_COL].copy()
y_test = df_test[TARGET_COL].copy()

target_distribution = build_target_distribution(df_train, df_test)

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)
print("Facturas compartidas:", int(len(set(train_ids["factura_id"]) & set(test_ids["factura_id"]))))
target_distribution


Shape X_train: (15735, 39)
Shape X_test: (3936, 39)
Facturas compartidas: 0


,split,target_mora,filas,porcentaje_filas,facturas
0,train,on_time,2646,16.8160,1773
1,train,+30,2962,18.8243,997
2,train,+60,4752,30.2002,870
3,train,+90,5375,34.1595,630
4,test,on_time,671,17.0478,444
5,test,+30,751,19.0803,249
6,test,+60,1213,30.8181,218
7,test,+90,1301,33.0539,157


## 6. Pipeline de preprocesamiento

El `ColumnTransformer` se ajusta solo con train. Las transformaciones aprendidas no usan informacion de test.


In [10]:
def build_preprocessor(feature_cols: list[str]) -> tuple[ColumnTransformer, dict]:
    numeric_log = [col for col in LOG_SKEWED_COLS if col in feature_cols]
    numeric_plain = [col for col in NUMERIC_PLAIN_COLS if col in feature_cols]
    binary = [col for col in BINARY_COLS if col in feature_cols]
    categorical = [col for col in CATEGORICAL_COLS if col in feature_cols]

    used = set(numeric_log + numeric_plain + binary + categorical)
    remainder_not_used = [col for col in feature_cols if col not in used]

    numeric_log_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", RobustScaler()),
    ])
    numeric_plain_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    binary_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="sin_gestion_previa")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num_log", numeric_log_pipe, numeric_log),
            ("num_plain", numeric_plain_pipe, numeric_plain),
            ("binary", binary_pipe, binary),
            ("cat", categorical_pipe, categorical),
        ],
        remainder="drop",
    )
    schema = {
        "numeric_log": numeric_log,
        "numeric_plain": numeric_plain,
        "binary": binary,
        "categorical": categorical,
        "remainder_not_used": remainder_not_used,
    }
    return preprocessor, schema


In [11]:
preprocessor, schema = build_preprocessor(feature_cols)
schema


{'numeric_log': ['monto',
  'monto_promedio_hist',
  'ratio_monto',
  'mora_promedio_hist',
  'mora_ultimo_tramo',
  'num_gestiones_factura',
  'dias_hasta_vence_pos',
  'dias_mora_observable',
  'num_no_contesta_cons',
  'num_promesas_rotas',
  'promesas_total',
  'dias_transcurridos_corte'],
 'numeric_plain': ['condicion_dias',
  'antiguedad_meses',
  'num_facturas_prev',
  'tasa_cumplimiento',
  'moras_consecutivas',
  'tasa_contacto_cliente',
  'tasa_cumpl_promesas',
  'friccion_contacto',
  'ratio_promesas_rotas',
  'intensidad_gestion',
  'dias_desde_ultima_gestion'],
 'binary': ['tiene_garantia',
  'tiene_disputa_activa',
  'tiene_promesa_activa',
  'sin_gestion_previa',
  'esta_vencida_al_corte',
  'cliente_nuevo',
  'sector_retail',
  'sector_manufactura',
  'sector_servicios',
  'sector_construccion',
  'sector_agro',
  'sector_tecnologia',
  'sector_salud',
  'sector_transporte'],
 'categorical': ['ultimo_resultado_enc'],
 'remainder_not_used': ['dias_hasta_vence']}

In [12]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
processed_feature_names = preprocessor.get_feature_names_out().tolist()

print("Shape X_train procesado:", X_train_proc.shape)
print("Shape X_test procesado:", X_test_proc.shape)
print("Features procesadas:", len(processed_feature_names))


Shape X_train procesado: (15735, 47)
Shape X_test procesado: (3936, 47)
Features procesadas: 47


## 7. Balanceamiento y augmentation

La estrategia base usa `class_weight` calculado solo con train. La augmentation queda como protocolo experimental y no modifica el dataset oficial.


In [13]:
def compute_train_class_weights(y_train: pd.Series) -> dict:
    classes = np.array([cls for cls in TARGET_CLASS_ORDER if cls in set(y_train)])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    return {str(cls): float(weight) for cls, weight in zip(classes, weights)}


def build_augmentation_protocol(df_train: pd.DataFrame, noise_scale: float = 0.01, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    minority_counts = df_train[TARGET_COL].value_counts()
    minority_class = minority_counts.idxmin()
    base = df_train[df_train[TARGET_COL] == minority_class].copy()
    if base.empty:
        return pd.DataFrame()

    numeric_aug_cols = [
        "monto", "ratio_monto", "mora_promedio_hist", "mora_ultimo_tramo",
        "monto_promedio_hist", "dias_hasta_vence", "num_promesas_rotas", "promesas_total",
    ]
    numeric_aug_cols = [col for col in numeric_aug_cols if col in base.columns]
    synthetic = base.sample(n=min(len(base), 200), replace=True, random_state=random_state).copy()

    for col in numeric_aug_cols:
        std = base[col].std(ddof=0)
        if pd.isna(std) or std == 0:
            continue
        synthetic[col] = synthetic[col] + rng.normal(0, noise_scale * std, size=len(synthetic))
        if (base[col] >= 0).all():
            synthetic[col] = synthetic[col].clip(lower=0)

    synthetic["es_augmented"] = 1
    return synthetic


In [14]:
class_weights = compute_train_class_weights(y_train)
synthetic_train_preview = build_augmentation_protocol(df_train)

class_weights


{'on_time': 1.4866780045351473,
 '+30': 1.3280722484807563,
 '+60': 0.8278093434343434,
 '+90': 0.7318604651162791}

In [15]:
synthetic_train_preview.head()


,factura_id,cliente_id,fecha_corte,monto,condicion_dias,antiguedad_meses,tiene_garantia,sector_retail,sector_manufactura,sector_servicios,...,sin_gestion_previa,dias_transcurridos_corte,esta_vencida_al_corte,dias_mora_observable,dias_hasta_vence_pos,cliente_nuevo,intensidad_gestion,friccion_contacto,ratio_promesas_rotas,es_augmented
6172,FAC001286,CLI0062,2023-07-12,7165.561560,90,48,0,0,0,0,...,0,15,0,0,75,0,0.0625,1.0,0.000000,1
9594,FAC004007,CLI0094,2024-07-04,6917.777364,45,16,0,0,0,1,...,1,0,0,0,45,0,0.0000,0.0,0.285714,1
8210,FAC004427,CLI0080,2024-08-31,42265.669048,30,10,0,0,1,0,...,1,0,0,0,30,0,0.0000,0.0,0.000000,1
8049,FAC003392,CLI0078,2024-04-07,30181.974711,60,43,1,0,0,0,...,1,0,0,0,60,0,0.0000,0.0,0.000000,1
12061,FAC000454,CLI0123,2023-03-09,7217.788787,60,7,1,0,0,0,...,1,0,0,0,60,0,0.0000,0.0,0.000000,1


## 8. Preparacion base para clustering de clientes

El componente de clustering trabaja a nivel `cliente_id`, no a nivel factura-corte. Esta celda construye una tabla base por cliente, sin target y sin escalado, para que la fase de evaluacion solo tenga que probar algoritmos y parametros.

In [ ]:
def build_sector_dominante(df: pd.DataFrame) -> pd.Series:
    sector_matrix = df[SECTOR_COLUMNS].fillna(0)
    active_counts = sector_matrix.gt(0).sum(axis=1)
    labels = sector_matrix.idxmax(axis=1).str.replace("sector_", "", regex=False)
    return labels.where(active_counts == 1, "sin_sector_unico")


def build_client_features_for_clustering(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    present_cont = [col for col in CLUSTERING_CONT_COLS if col in df.columns]
    present_ratio = [col for col in CLUSTERING_RATIO_COLS if col in df.columns]
    present_count = [col for col in CLUSTERING_COUNT_COLS if col in df.columns]
    present_binary = [col for col in CLUSTERING_BINARY_COLS if col in df.columns]

    agg_dict = {}
    for col in present_cont:
        agg_dict[col] = ["mean", "max"]
    for col in present_ratio:
        agg_dict[col] = ["mean"]
    for col in present_count:
        agg_dict[col] = ["sum", "mean"]
    for col in present_binary:
        agg_dict[col] = ["mean"]

    client_features = df.groupby("cliente_id").agg(agg_dict)
    client_features.columns = ["_".join(col).strip() for col in client_features.columns]
    client_features = client_features.reset_index()

    client_counts = (
        df.groupby("cliente_id")
        .agg(
            n_facturas_total=("factura_id", "nunique"),
            n_cortes_total=("factura_id", "size"),
        )
        .reset_index()
    )
    client_features = client_features.merge(client_counts, on="cliente_id", how="left")

    df_sector = df[["cliente_id", *SECTOR_COLUMNS]].copy()
    df_sector["sector_dominante"] = build_sector_dominante(df_sector)
    sector_modal = (
        df_sector.groupby("cliente_id")["sector_dominante"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "sin_sector_unico")
        .reset_index()
        .rename(columns={"sector_dominante": "sector_dominante_modal"})
    )
    client_features = client_features.merge(sector_modal, on="cliente_id", how="left")

    feature_cols_clustering = [
        col for col in client_features.columns
        if col not in ["cliente_id", "sector_dominante_modal"]
    ]

    for col in feature_cols_clustering:
        if col.startswith(tuple(f"{base}_" for base in present_count)):
            client_features[col] = client_features[col].fillna(0)
        elif col.startswith(tuple(f"{base}_" for base in present_binary)):
            client_features[col] = client_features[col].fillna(0)
        else:
            client_features[col] = client_features[col].fillna(client_features[col].median())

    column_summary = []
    for col in client_features.columns:
        if col == "cliente_id":
            role = "trazabilidad"
        elif col == "sector_dominante_modal":
            role = "perfil_interpretativo"
        else:
            role = "feature_clustering"
        column_summary.append({
            "columna": col,
            "rol": role,
            "dtype": str(client_features[col].dtype),
            "nulos": int(client_features[col].isna().sum()),
            "valores_unicos": int(client_features[col].nunique(dropna=True)),
        })

    features_selected = pd.DataFrame({"feature": feature_cols_clustering})
    columns_summary = pd.DataFrame(column_summary)
    return client_features, features_selected, columns_summary


client_features_clustering_base, client_clustering_features_selected, client_clustering_columns_summary = build_client_features_for_clustering(df_prepared)

print("Shape dataset base clustering:", client_features_clustering_base.shape)
client_features_clustering_base.head()


## 9. Exportacion de artefactos tecnicos

El notebook exporta datos, split oficial, metadata, resumen de columnas, revision de redundancias y pipeline serializado. No genera reportes narrativos.


In [16]:
def build_column_summary(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    feature_set = set(feature_cols)
    rows = []
    for col in df.columns:
        if col == TARGET_COL:
            role = "target"
        elif col in ["factura_id", "cliente_id", "fecha_corte"]:
            role = "trazabilidad"
        elif col in feature_set:
            role = "feature"
        else:
            role = "no_usada"
        rows.append({
            "columna": col,
            "rol": role,
            "dtype": str(df[col].dtype),
            "nulos": int(df[col].isna().sum()),
            "valores_unicos": int(df[col].nunique(dropna=True)),
        })
    return pd.DataFrame(rows)


def export_artifacts(
    df_final: pd.DataFrame,
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    train_ids: pd.DataFrame,
    test_ids: pd.DataFrame,
    schema: dict,
    validations: dict,
    validation_checklist: pd.DataFrame,
    split_integrity_check: pd.DataFrame,
    cleaning_summary: dict,
    redundancy_review: pd.DataFrame,
    class_weights: dict,
    outlier_table: pd.DataFrame,
    target_distribution: pd.DataFrame,
    feature_cols: list[str],
    preprocessor: ColumnTransformer,
    processed_feature_names: list[str],
    synthetic_train_preview: pd.DataFrame,
    client_features_clustering_base: pd.DataFrame,
    client_clustering_features_selected: pd.DataFrame,
    client_clustering_columns_summary: pd.DataFrame,
) -> dict:
    column_summary = build_column_summary(df_final, feature_cols)

    df_final.to_csv(OUTPUT_DIR / "features_ml_prepared.csv", index=False)
    train_ids.to_csv(OUTPUT_DIR / "train_facturas_ids.csv", index=False)
    test_ids.to_csv(OUTPUT_DIR / "test_facturas_ids.csv", index=False)
    outlier_table.to_csv(OUTPUT_DIR / "outlier_summary.csv", index=False)
    validation_checklist.to_csv(OUTPUT_DIR / "preprocessing_validation_checklist.csv", index=False)
    split_integrity_check.to_csv(OUTPUT_DIR / "split_integrity_check.csv", index=False)
    target_distribution.to_csv(OUTPUT_DIR / "target_distribution_train_test.csv", index=False)
    redundancy_review.to_csv(OUTPUT_DIR / "redundancy_review.csv", index=False)
    column_summary.to_csv(OUTPUT_DIR / "prepared_columns_summary.csv", index=False)
    pd.DataFrame({"feature": feature_cols}).to_csv(OUTPUT_DIR / "features_selected.csv", index=False)
    pd.DataFrame({"processed_feature": processed_feature_names}).to_csv(OUTPUT_DIR / "processed_feature_names.csv", index=False)
    client_features_clustering_base.to_csv(OUTPUT_DIR / "client_features_clustering_base.csv", index=False)
    client_clustering_features_selected.to_csv(OUTPUT_DIR / "client_clustering_features_selected.csv", index=False)
    client_clustering_columns_summary.to_csv(OUTPUT_DIR / "client_clustering_columns_summary.csv", index=False)
    joblib.dump(preprocessor, OUTPUT_DIR / "preprocessing_pipeline.joblib")

    metadata = {
        "input_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "output_dir": str(OUTPUT_DIR.relative_to(PROJECT_ROOT)),
        "shape_input": validations["shape"],
        "shape_final": list(df_final.shape),
        "train_rows": int(len(df_train)),
        "test_rows": int(len(df_test)),
        "train_facturas": int(train_ids["factura_id"].nunique()),
        "test_facturas": int(test_ids["factura_id"].nunique()),
        "facturas_overlap_train_test": int(len(set(train_ids["factura_id"]) & set(test_ids["factura_id"]))),
        "feature_count_unprocessed": len(feature_cols),
        "feature_count_processed": len(processed_feature_names),
        "client_clustering_rows": int(len(client_features_clustering_base)),
        "client_clustering_feature_count": int(len(client_clustering_features_selected)),
        "excluded_from_features": ID_AND_LABEL_COLUMNS,
        "simulation_only_excluded": SIMULATION_ONLY_COLUMNS,
        "validations": validations,
        "cleaning_summary": cleaning_summary,
        "schema": schema,
        "class_weights": class_weights,
        "augmentation_protocol": {
            "applied_to_main_dataset": False,
            "reason": "Se conserva como protocolo experimental; no se mezcla con test ni con el dataset base.",
            "synthetic_rows_preview_generated_on_train": int(len(synthetic_train_preview)),
        },
    }
    with open(OUTPUT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)
    return metadata


In [17]:
metadata = export_artifacts(
    df_final=df_prepared,
    df_train=df_train,
    df_test=df_test,
    train_ids=train_ids,
    test_ids=test_ids,
    schema=schema,
    validations=validations,
    validation_checklist=validation_checklist,
    split_integrity_check=split_integrity_check,
    cleaning_summary=cleaning_summary,
    redundancy_review=redundancy_review,
    class_weights=class_weights,
    outlier_table=outlier_table,
    target_distribution=target_distribution,
    feature_cols=feature_cols,
    preprocessor=preprocessor,
    processed_feature_names=processed_feature_names,
    synthetic_train_preview=synthetic_train_preview,
    client_features_clustering_base=client_features_clustering_base,
    client_clustering_features_selected=client_clustering_features_selected,
    client_clustering_columns_summary=client_clustering_columns_summary,
)

print("Artefactos exportados en:", OUTPUT_DIR.relative_to(PROJECT_ROOT))


Artefactos exportados en: 03_preparacion\outputs


## 10. Resumen final

La ultima celda resume las salidas tecnicas y las validaciones criticas de continuidad.


In [18]:
output_files = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())
summary = {
    "shape_final": metadata["shape_final"],
    "train_facturas": metadata["train_facturas"],
    "test_facturas": metadata["test_facturas"],
    "overlap_facturas": metadata["facturas_overlap_train_test"],
    "feature_count_unprocessed": metadata["feature_count_unprocessed"],
    "feature_count_processed": metadata["feature_count_processed"],
    "client_clustering_rows": metadata["client_clustering_rows"],
    "client_clustering_feature_count": metadata["client_clustering_feature_count"],
    "validation_failures": int((validation_checklist["estado"] != "Cumple").sum()),
    "split_failures": int((split_integrity_check["estado"] != "Cumple").sum()),
    "outputs": output_files,
}
summary


{'shape_final': [19671, 43],
 'train_facturas': 4270,
 'test_facturas': 1068,
 'overlap_facturas': 0,
 'feature_count_unprocessed': 39,
 'feature_count_processed': 47,
 'validation_failures': 0,
 'split_failures': 0,
 'outputs': ['features_ml_prepared.csv',
  'features_selected.csv',
  'outlier_summary.csv',
  'prepared_columns_summary.csv',
  'preprocessing_metadata.json',
  'preprocessing_pipeline.joblib',
  'preprocessing_validation_checklist.csv',
  'processed_feature_names.csv',
  'redundancy_review.csv',
  'split_integrity_check.csv',
  'target_distribution_train_test.csv',
  'test_facturas_ids.csv',
  'train_facturas_ids.csv']}